---

# 🎁 도전과제

이론 실습은 끝났습니다. 이제 배운 걸로 **실무에서 자주 만나는** 작업을 직접 만들어봅시다.

## 도전과제 1. 회의록 요약 — system 메시지 선택하기

긴 회의록을 받아서 3줄 요약을 돌려주는 비서를 만들어야 합니다.  
빈칸을 채우는 게 아니라 **3가지 system 메시지 후보 중 어느 게 가장 좋은가** 판단해보세요.

- 셋 다 돌려보고, 결과를 비교한 뒤 *근거를 들어* 한 개를 고릅니다.
- 이게 실무에서 프롬프트를 설계하는 진짜 방식이에요.

In [11]:
# 🔧 LLM 호출 헬퍼 함수 — 앞으로 계속 사용합니다
def ask_llm(messages, temperature=0.3, max_tokens=500):
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


In [6]:
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")

✅ 로컬 Jupyter 환경입니다.


In [7]:
# 3️⃣ API 키 불러오기 (환경에 따라 자동 분기)
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()  # .env 파일에서 자동으로 불러옴

# 키가 잘 들어왔는지 확인 (앞 7글자만 출력 — 전체 노출은 위험)
api_key = os.environ.get("OPENAI_API_KEY", "")
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")


API 키 로드: sk-proj...


In [8]:
# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")


✅ OpenAI 클라이언트 준비 완료


In [9]:
# 🎯 후보 system 메시지 3개 — 셋 다 돌려보고 비교
candidates = {
    "A": "회의록을 요약하세요.",
    "B": "당신은 회의록 요약 비서입니다. 3줄로 요약하세요.",
    "C": ("당신은 회의록 요약 비서입니다. 정확히 3줄로 요약하되, "
          "각 줄은 (결정사항) / (담당자) / (마감일) 형식을 따르세요. "
          "회의록에 없는 내용은 절대 추측하지 마세요."),
}

sample_notes = """
[2026-05-14 팀 미팅]
- 김PM: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안. 이유는 평가 데이터 부족.
- 이개발: 6월 15일이면 가능. 다만 RAG 평가셋 100건 추가 수집 필요.
- 박마케팅: 출시 연기 시 광고 일정도 조정해야 함. 다음 주 월요일까지 결정 필요.
"""

# 세 후보를 모두 실행해서 비교
for key, system_msg in candidates.items():
    result = ask_llm([
        {"role": "system", "content": system_msg},
        {"role": "user", "content": sample_notes},
    ], temperature=0)
    print(f"=== 후보 {key} ===")
    print(result)
    print()

=== 후보 A ===
('**회의록 요약 (2026-05-14 팀 미팅)**\n\n- **김PM**: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안, 이유는 평가 데이터 부족.\n- **이개발**: 6월 15일 출시 가능하나, RAG 평가셋 100건 추가 수집 필요.\n- **박마케팅**: 출시 연기 시 광고 일정 조정 필요, 다음 주 월요일까지 결정 필요.', 233)

=== 후보 B ===
('김PM은 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기할 것을 제안하며, 그 이유는 평가 데이터 부족이라고 설명했다. 이개발은 6월 15일 출시가 가능하나 RAG 평가셋 100건 추가 수집이 필요하다고 언급했다. 박마케팅은 출시 연기에 따라 광고 일정 조정이 필요하며, 다음 주 월요일까지 결정을 내려야 한다고 강조했다.', 241)

=== 후보 C ===
('(신규 챗봇 출시일을 6월 15일로 연기) / (김PM, 이개발, 박마케팅) / (다음 주 월요일)', 219)



### 🤔 판단해보기

세 결과를 나란히 놓고 *2줄로* 답해보세요 (이 셀 아래에 마크다운 셀 하나 더 추가해서 적어도 좋아요):

1. 실무에서 가장 쓸 만한 건 어느 것인가? (A / B / C)
2. C가 좋다면, A나 B에 *없는* 어떤 요소 때문인가? (혹은 B가 좋다면 그 이유는?)

> 💡 **이 연습의 진짜 의미**: AI 시대의 핵심 스킬은 "프롬프트를 처음부터 잘 쓰는 것"보다  
> *여러 후보를 빠르게 비교해서 좋은 걸 고르는 판단력*입니다.

1. C
2. **출력 형식**을 고정해놨기 때문에 사용자가 원하는 형태로 응답이 나온다.

## 도전과제 2. 고객 문의 자동 분류기

고객 문의 한 줄을 받아서 **`결제`, `배송`, `환불`, `기타`** 중 하나로 분류하세요.

이제 여러분은 *왜* `temperature=0`, `max_tokens=10`이 필요한지 압니다 (위의 🐛 버그 셀 기억나죠?).

**힌트**:
- `system` 메시지에 "다음 4개 중 하나로만 답하세요"라고 명시
- `temperature=0` (일관성 필수), `max_tokens=10` (단어 하나면 충분)

In [12]:
def classify_inquiry(text: str) -> str:
    """고객 문의를 4개 카테고리 중 하나로 분류"""
    # 👇 여기를 채워보세요
    messages = [
        {"role": "system", "content": "다음 4개 중 하나로만 답하세요. 결제, 배송, 환불, 기타"},  # ← 빈칸
        {"role": "user", "content": text},
    ]
    return ask_llm(messages, temperature=0, max_tokens=10)


# 🎯 실행 *전* 예측: 아래 4개 문의가 어떻게 분류될지 머릿속으로 먼저 답을 적어보세요
test_cases = [
    "어제 주문했는데 카드 결제가 두 번 되었어요.",       # 예측: ____결제
    "택배가 일주일째 안 와요.",                          # 예측: ____배송
    "사이즈가 안 맞아서 반품하고 싶어요.",                # 예측: ____환불
    "영업시간이 어떻게 되나요?",                         # 예측: ____기타
]

# 이제 실행해서 예측과 비교
for text in test_cases:
    category = classify_inquiry(text)
    print(f"[{category:>4}] {text}")

[  환불] 어제 주문했는데 카드 결제가 두 번 되었어요.
[  배송] 택배가 일주일째 안 와요.
[  환불] 사이즈가 안 맞아서 반품하고 싶어요.
[  기타] 영업시간이 어떻게 되나요?


✅ **결과 확인**: 각 문의가 알맞은 카테고리로 분류됐나요?

> 💡 이 패턴이 바로 실무에서 가장 많이 쓰이는 **LLM 활용 패턴 1순위**입니다.  
> 분류, 추출, 요약 — 이 3개만 자유자재로 다뤄도 회사에서 쓸 챗봇·자동화 절반은 만들 수 있어요.

---

## 🏁 모두 풀었다면

축하합니다! Day 1 Lesson 1 완료입니다 🎉  
다음 노트북 `Day1_Lesson2_LangChain.ipynb`에서 만나요.
